# Debug SBI Training Data

Use this notebook to inspect the exact arrays passed to SBI: the parameter tensor (`theta`), the transformed simulator outputs (`x`), and the transformed observation used for posterior inference (`default_x`).

In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

Set these to the output directory and round you want to inspect. `round_index=0` means round 1.

In [ ]:
output_dir = Path("output_FlopPITy")
round_index = 0

round_dir = output_dir / "rounds" / f"round_{round_index:03d}"
sbi_path = round_dir / "sbi_data.npz"
training_path = round_dir / "training_data.npz"

print(sbi_path)
print("exists:", sbi_path.exists())

In [ ]:
with np.load(sbi_path, allow_pickle=False) as archive:
    theta = archive["theta"]
    x = archive["x"]
    default_x = archive["default_x"]
    metadata = json.loads(archive["metadata"].item())

print(json.dumps(metadata, indent=2))
print("theta", theta.shape, theta.dtype, "finite", np.isfinite(theta).all())
print("x", x.shape, x.dtype, "finite", np.isfinite(x).all())
print("default_x", default_x.shape, default_x.dtype, "finite", np.isfinite(default_x).all())

## Transformed Spectra Seen By SBI

The shaded region shows the 5th to 95th percentile range of the transformed training data at each feature. The black line is the transformed observation (`default_x`) used for inference.

In [ ]:
feature = np.arange(x.shape[1])
x_p05, x_p50, x_p95 = np.percentile(x, [5, 50, 95], axis=0)
default_line = default_x.reshape(-1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(feature, x_p05, x_p95, alpha=0.25, label="training x: 5-95%")
ax.plot(feature, x_p50, label="training x: median")
ax.plot(feature, default_line, color="black", linewidth=1.5, label="default_x")
ax.set_xlabel("feature index")
ax.set_ylabel("transformed value")
ax.legend()
plt.show()

## Where The Observation Falls Relative To Training Data

This is often the fastest check for a mismatch. Values near 0 mean the transformed observation is around the training median; values with large absolute magnitude are outside the feature-wise training spread.

In [ ]:
x_mean = np.mean(x, axis=0)
x_std = np.std(x, axis=0, ddof=1)
x_std = np.clip(x_std, 1e-12, None)
default_z = (default_line - x_mean) / x_std

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(feature, default_z, color="tab:red")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].axhline(3, color="gray", linestyle="--", linewidth=1)
axes[0].axhline(-3, color="gray", linestyle="--", linewidth=1)
axes[0].set_xlabel("feature index")
axes[0].set_ylabel("(default_x - mean(x)) / std(x)")

axes[1].hist(default_z[np.isfinite(default_z)], bins=50, color="tab:red", alpha=0.75)
axes[1].set_xlabel("feature-wise z score")
axes[1].set_ylabel("count")
plt.tight_layout()
plt.show()

print("max |z|:", np.nanmax(np.abs(default_z)))
print("features with |z| > 3:", int(np.sum(np.abs(default_z) > 3)))

## Heatmap Of Training Inputs

This plots a subset of rows from the exact `x` array passed to SBI.

In [ ]:
n_show = min(200, x.shape[0])
rng = np.random.default_rng(123)
rows = np.sort(rng.choice(x.shape[0], size=n_show, replace=False))

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(x[rows], aspect="auto", interpolation="nearest", cmap="viridis")
ax.set_xlabel("feature index")
ax.set_ylabel("sample index subset")
fig.colorbar(im, ax=ax, label="transformed x")
plt.show()

## Parameters Paired With SBI Inputs

In [ ]:
parameter_names = metadata.get("parameter_names") or [f"theta_{i}" for i in range(theta.shape[1])]
n_cols = min(theta.shape[1], 4)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 3), squeeze=False)
for i, ax in enumerate(axes[0]):
    ax.hist(theta[:, i], bins=40, alpha=0.8)
    ax.set_title(parameter_names[i])
    ax.set_xlabel("value")
    ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## Optional: Inspect Raw Saved Training Data

If `training_data.npz` exists for this round, this prints the stored raw/postprocessed simulator arrays. These are not what SBI sees directly, but they are useful for comparing against `x` after preprocessing.

In [ ]:
if training_path.exists():
    with np.load(training_path, allow_pickle=False) as archive:
        training_metadata = json.loads(archive["metadata"].item())
        print(json.dumps(training_metadata, indent=2))
        print("arrays:", archive.files)
        for name in archive.files:
            if name != "metadata":
                print(name, archive[name].shape, archive[name].dtype, "finite", np.isfinite(archive[name]).all() if np.issubdtype(archive[name].dtype, np.number) else "n/a")
else:
    print("No training_data.npz found for this round:", training_path)